# Modul ADV-04: Advanced Deep Recognition & Attention Mechanisms
## Hands-on Laboratory: Scaled Attention, Deformable Sampling, GIoU, and DETR Loss

**Author:** Aji Pamungkas Tri Nurcahyo  
**Institution:** Institut Teknologi Sumatera (ITERA)  
**Level:** L3 Advanced Curriculum / Research Track R01 Preparation

Laboratorium komputasi ini mengimplementasikan fondasi matematika pengenalan visual canggih:
1. **Scaled Dot-Product Attention:** Normalisasi Softmax dan dimensionalitas matriks $Q, K, V$.
2. **Deformable Attention 2D:** Interpolasi bilinear sub-piksel kontinu pada titik sampel terdeformasi.
3. **Metrik Kotak Pembatas:** IoU standar vs Generalized IoU (GIoU) dengan penanganan kotak tanpa irisan.
4. **Focal Loss:** Modulasi gradien $(1 - p_t)^\gamma$ untuk ketidakseimbangan kelas ekstrem.
5. **DETR Bipartite Matching:** Matriks biaya penugasan global dan pemecahan Kuhn-Munkres.

In [ ]:
import numpy as np

print("NumPy Version:", np.__version__)
print("Environment initialized successfully for ADV-04 Deep Recognition Lab.")

In [ ]:
def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray):
    dk = Q.shape[-1]
    scale = 1.0 / np.sqrt(max(1, dk))
    scores = (Q @ K.T) * scale
    
    # Numerically stable row-wise softmax
    scores_max = np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scores - scores_max)
    weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    output = weights @ V
    return weights, output

# Uji coba matriks Query, Key, Value
Q = np.array([[1.0, 0.0, 0.5], [0.0, 1.0, 0.2]], dtype=np.float64)
K = np.array([[1.0, 0.0, 0.5], [0.0, 1.0, 0.2], [0.5, 0.5, 0.0]], dtype=np.float64)
V = np.array([[10.0, 0.0], [0.0, 20.0], [5.0, 5.0]], dtype=np.float64)

w, out = scaled_dot_product_attention(Q, K, V)
print("Attention Weights Matrix:\n", np.round(w, 4))
print("Row Sums:", np.round(np.sum(w, axis=-1), 4))
print("Context Output:\n", np.round(out, 4))

In [ ]:
def bilinear_interpolate_2d(grid: np.ndarray, x: float, y: float) -> float:
    H, W = grid.shape
    xc = np.clip(x, 0.0, W - 1.0)
    yc = np.clip(y, 0.0, H - 1.0)
    x0, x1 = int(np.floor(xc)), min(W - 1, int(np.floor(xc)) + 1)
    y0, y1 = int(np.floor(yc)), min(H - 1, int(np.floor(yc)) + 1)
    wx, wy = xc - x0, yc - y0
    top = (1.0 - wx) * grid[y0, x0] + wx * grid[y0, x1]
    bottom = (1.0 - wx) * grid[y1, x0] + wx * grid[y1, x1]
    return float((1.0 - wy) * top + wy * bottom)

feature_grid = np.array([
    [10.0, 20.0, 15.0],
    [30.0, 50.0, 40.0],
    [25.0, 35.0, 30.0]
], dtype=np.float64)

# Titik sub-piksel kontinu (0.5, 0.5)
sampled = bilinear_interpolate_2d(feature_grid, 0.5, 0.5)
print(f"Nilai interpolasi sub-piksel pada (0.5, 0.5): {sampled:.2f}")

In [ ]:
def compute_iou_giou(boxA, boxB):
    ax1, ay1, ax2, ay2 = boxA
    bx1, by1, bx2, by2 = boxB
    
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    intersection = iw * ih
    
    areaA = (ax2 - ax1) * (ay2 - ay1)
    areaB = (bx2 - bx1) * (by2 - by1)
    union = areaA + areaB - intersection
    iou = intersection / union if union > 0 else 0.0
    
    # Smallest enclosing convex box C
    cx1, cy1 = min(ax1, bx1), min(ay1, by1)
    cx2, cy2 = max(ax2, bx2), max(ay2, by2)
    areaC = max(1e-8, (cx2 - cx1) * (cy2 - cy1))
    
    giou = iou - (areaC - union) / areaC
    return iou, giou

box1 = [0, 0, 10, 10]
box2 = [5, 0, 15, 10]
box3 = [20, 20, 30, 30] # Disjoint

iou_ov, giou_ov = compute_iou_giou(box1, box2)
iou_dj, giou_dj = compute_iou_giou(box1, box3)

print(f"Tumpang Tindih 50% -> IoU: {iou_ov:.4f}, GIoU: {giou_ov:.4f}")
print(f"Tanpa Tumpang Tindih (Disjoint) -> IoU: {iou_dj:.4f}, GIoU: {giou_dj:.4f} (Gradien non-nol)")

In [ ]:
def focal_loss(p: float, y: int, alpha: float = 0.25, gamma: float = 2.0) -> float:
    pt = p if y == 1 else (1.0 - p)
    pt = max(1e-7, min(1.0 - 1e-7, pt))
    alpha_t = alpha if y == 1 else (1.0 - alpha)
    return float(-alpha_t * ((1.0 - pt) ** gamma) * np.log(pt))

print("Perbandingan Penalti Focal Loss vs Cross-Entropy:")
for prob in [0.99, 0.90, 0.50, 0.10, 0.01]:
    fl = focal_loss(prob, y=1, gamma=2.0)
    ce = -np.log(prob)
    print(f"Keyakinan p={prob:.2f} | Focal Loss: {fl:.6f} | Standard CE: {ce:.6f}")

In [ ]:
def compute_detr_matching_cost(pred_box, pred_prob, target_box, target_class, pred_class):
    class_cost = -1.0 * pred_prob if pred_class == target_class else 0.0
    l1_cost = 5.0 * np.sum(np.abs(np.array(pred_box) - np.array(target_box)))
    _, giou = compute_iou_giou(pred_box, target_box)
    giou_cost = 2.0 * (1.0 - giou)
    return class_cost + l1_cost + giou_cost

# 2 Prediksi Query vs 2 Target Ground Truth
queries = [
    {"box": [0.1, 0.1, 0.3, 0.3], "prob": 0.92, "class": 0},
    {"box": [0.7, 0.7, 0.9, 0.9], "prob": 0.88, "class": 1}
]
targets = [
    {"box": [0.12, 0.11, 0.31, 0.29], "class": 0},
    {"box": [0.69, 0.71, 0.88, 0.91], "class": 1}
]

cost_mat = np.zeros((2, 2))
for q in range(2):
    for t in range(2):
        cost_mat[q, t] = compute_detr_matching_cost(
            queries[q]["box"], queries[q]["prob"],
            targets[t]["box"], targets[t]["class"], queries[q]["class"]
        )

print("Matriks Biaya Kecocokan DETR (Cost Matrix):\n", np.round(cost_mat, 4))
print("Penugasan Optimal: Query 0 -> Target 0, Query 1 -> Target 1")

### Kesimpulan Laboratorium & Keterkaitan Riset (Track R01)
1. **Deformable Attention** mengurangi komputasi kuadratik dengan memfokuskan sampel representasi hanya pada $K$ titik kunci terdeformasi.
2. **GIoU** menyediakan gradien perbaikan yang kontinu bahkan ketika kotak pembatas prediksi dan ground truth tidak beririsan.
3. **Focal Loss** menekan gradien latar belakang mudah sebesar $>1000\times$, memungkinkan pelatihan detektor satu tahap yang stabil.
4. Formulasi ini menjadi landasan implementasi detektor SOTA (Deformable DETR, RT-DETR) pada **Research Track R01** (MS COCO 2017).